# Synthesis Cost Comparison

24 synthesis routes ranked by cost, time, safety, and feasibility.
Equipment overlap analysis: which equipment serves the most targets?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import json

csv_path = '../showcase/outputs/synthesis_all_targets.csv'
json_path = '../showcase/outputs/synthesis_comparison.json'

if not os.path.exists(csv_path):
    print('Generating data...')
    from showcase.synthesis_comparison import main
    main()

df = pd.read_csv(csv_path)
with open(json_path) as f:
    data = json.load(f)

print(f'Total targets: {data["total_targets"]}')
print(f'Total routes: {data["total_routes"]}')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Composite score ranking
fig, ax = plt.subplots(figsize=(12, 6))
sorted_df = df.sort_values('composite', ascending=True)

colors = plt.cm.RdYlGn(sorted_df['composite'] / sorted_df['composite'].max())
ax.barh(range(len(sorted_df)), sorted_df['composite'], color=colors, edgecolor='black')
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels([f"{r['target']} ({r['route_name']})" for _, r in sorted_df.iterrows()],
                   fontsize=7)
ax.set_xlabel('Composite Score')
ax.set_title('All Synthesis Routes Ranked by Composite Score')
ax.axvline(0.5, color='red', linestyle='--', label='Low viability threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cost vs Time scatter plot
summaries = [s for s in data['target_summaries'] if 'error' not in s]
costs = [s['precursor_cost_usd'] for s in summaries]
times = [s['total_time_hours'] for s in summaries]
names = [s['target'] for s in summaries]
composites = [s['best_composite'] or 0 for s in summaries]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(costs, times, c=composites, cmap='RdYlGn',
                     s=100, edgecolors='black', vmin=0.4, vmax=1.0)
plt.colorbar(scatter, label='Best Composite Score')

for i, name in enumerate(names):
    ax.annotate(name, (costs[i], times[i]), fontsize=7,
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Precursor Cost ($/g)')
ax.set_ylabel('Total Time (hours)')
ax.set_title('Synthesis Route: Cost vs Time (color = composite score)')
plt.tight_layout()
plt.show()

In [ ]:
# Safety rating distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['safety_score'], bins=10, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Safety Score (0=hazardous, 1=safe)')
ax.set_ylabel('Number of Routes')
ax.set_title('Safety Score Distribution Across All Routes')
ax.axvline(df['safety_score'].mean(), color='red', linestyle='--',
           label=f'Mean = {df["safety_score"].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Equipment utilization matrix
equip = data.get('equipment_analysis', {}).get('equipment_frequency', {})
equip_targets = data.get('equipment_analysis', {}).get('equipment_targets', {})

if equip:
    sorted_equip = sorted(equip.items(), key=lambda x: x[1], reverse=True)
    eq_names = [e[0] for e in sorted_equip]
    eq_counts = [e[1] for e in sorted_equip]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(eq_names[::-1], eq_counts[::-1], color='steelblue', edgecolor='black')
    ax.set_xlabel('Number of Target Materials Served')
    ax.set_title('Equipment Utilization: Which Equipment Is Most Versatile?')

    for i, (name, count) in enumerate(zip(eq_names[::-1], eq_counts[::-1])):
        targets = equip_targets.get(name, [])
        ax.text(count + 0.1, i, ', '.join(targets[:3]) + ('...' if len(targets) > 3 else ''),
                va='center', fontsize=7)

    plt.tight_layout()
    plt.show()

    print('\nWith stirrer + furnace + oven + ball_mill, you can make:')
    core_equip = {'stirrer', 'furnace', 'oven', 'ball_mill'}
    for s in summaries:
        needed = set(s.get('equipment_needed', []))
        if needed <= core_equip:
            print(f'  {s["target"]} (composite={s["best_composite"]:.3f})')